# Exploratory Data Analysis: Car Price Prediction

**Capstone Project -- Car Selling Price Prediction**

This notebook performs a thorough exploratory data analysis (EDA) on the cleaned CarDekho dataset. The goal is to understand the structure of the data, examine relationships between features and the target variable (`Selling_Price`), and surface insights that will inform feature engineering and model selection in later stages.

**Dataset overview:**
- **Source:** CarDekho (cleaned and preprocessed)
- **Records:** ~300 used-car listings
- **Target:** `Selling_Price` (in lakhs)
- **Features:** Brand, Present_Price, Kms_Driven, Fuel_Type, Seller_Type, Transmission, Owner, Car_Age

**Sections:**
1. Load Data
2. Target Variable Analysis
3. Numeric Features
4. Categorical Features
5. Key Insights

## 1. Load Data and Initial Inspection

We start by loading the cleaned dataset and running basic diagnostics: shape, first few rows, summary statistics, and column types. This gives us a quick sanity check before diving deeper.

In [ ]:
# -- Imports and configuration --
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
})

print("Libraries loaded successfully.")

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)

In [ ]:
# Column types, non-null counts, and memory usage
df.info()

In [ ]:
# Check for any remaining missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values -- dataset is clean.")

---
## 2. Target Variable Analysis

The target variable is `Selling_Price` (in lakhs). Before modelling, we need to understand its distribution. Right-skewed targets are common in price data and may benefit from a log transformation to stabilise variance and improve model performance.

In [ ]:
# Distribution of Selling_Price (original and log-transformed)
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
sns.histplot(df['Selling_Price'], kde=True, bins=30, color='steelblue', ax=axes[0])
skew_val = df['Selling_Price'].skew()
axes[0].set_title(f'Selling Price Distribution\n(Skewness: {skew_val:.2f})')
axes[0].set_xlabel('Selling Price (Lakhs)')
axes[0].set_ylabel('Count')

# Log-transformed distribution
log_price = np.log1p(df['Selling_Price'])
sns.histplot(log_price, kde=True, bins=30, color='coral', ax=axes[1])
log_skew = log_price.skew()
axes[1].set_title(f'Log-Transformed Selling Price\n(Skewness: {log_skew:.2f})')
axes[1].set_xlabel('log(1 + Selling Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Original skewness:        {skew_val:.3f}")
print(f"Log-transformed skewness: {log_skew:.3f}")
print(f"\\nThe log transform substantially reduces skewness, making the distribution closer to normal.")

**Observation:** Selling prices are right-skewed -- most cars sell at lower price points while a few premium vehicles push the tail out. The log transformation pulls the distribution much closer to normal, which is beneficial for linear models and regularisation-based approaches.

---
## 3. Numeric Features

We now examine how the numeric features relate to each other and to the target. A correlation heatmap gives the big picture, followed by scatter plots for the three most important numeric predictors.

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Correlation Heatmap -- Numeric Features')
plt.tight_layout()
plt.show()

The heatmap highlights which features have the strongest linear relationships with `Selling_Price`. Let us now visualise these individual relationships with scatter plots.

In [ ]:
# Scatter plots: Selling_Price vs key numeric predictors
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Present_Price vs Selling_Price
sns.scatterplot(data=df, x='Present_Price', y='Selling_Price', alpha=0.6,
                color='steelblue', ax=axes[0])
axes[0].set_title('Selling Price vs Present Price')
axes[0].set_xlabel('Present Price (Lakhs)')
axes[0].set_ylabel('Selling Price (Lakhs)')

# Car_Age vs Selling_Price
sns.scatterplot(data=df, x='Car_Age', y='Selling_Price', alpha=0.6,
                color='coral', ax=axes[1])
axes[1].set_title('Selling Price vs Car Age')
axes[1].set_xlabel('Car Age (Years)')
axes[1].set_ylabel('Selling Price (Lakhs)')

# Kms_Driven vs Selling_Price
sns.scatterplot(data=df, x='Kms_Driven', y='Selling_Price', alpha=0.6,
                color='seagreen', ax=axes[2])
axes[2].set_title('Selling Price vs Kilometres Driven')
axes[2].set_xlabel('Kilometres Driven')
axes[2].set_ylabel('Selling Price (Lakhs)')

plt.tight_layout()
plt.show()

**Observations:**
- **Present Price** shows a strong positive linear relationship with Selling Price -- cars with higher showroom prices retain higher resale value.
- **Car Age** shows a negative relationship -- older cars sell for less, as expected through depreciation.
- **Kilometres Driven** has a weaker, noisier negative relationship -- high-mileage cars tend to sell cheaper, but the effect is less pronounced than age or present price.

---
## 4. Categorical Features

Next we explore how the categorical variables -- Fuel_Type, Transmission, Seller_Type, and Brand -- relate to the selling price. Box plots reveal price distributions within each category, and bar charts show the sample sizes per category.

In [ ]:
# Box plots: Selling Price by categorical features
cat_features = ['Fuel_Type', 'Transmission', 'Seller_Type']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(cat_features):
    order = df.groupby(col)['Selling_Price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='Selling_Price', order=order,
                palette='Set2', ax=axes[i])
    axes[i].set_title(f'Selling Price by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Selling Price (Lakhs)')

plt.tight_layout()
plt.show()

In [ ]:
# Box plot: Selling Price by Brand (top 10 brands by count for readability)
top_brands = df['Brand'].value_counts().head(10).index
df_top = df[df['Brand'].isin(top_brands)]
brand_order = df_top.groupby('Brand')['Selling_Price'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_top, x='Brand', y='Selling_Price', order=brand_order,
            palette='viridis', ax=ax)
ax.set_title('Selling Price by Brand (Top 10 by Volume)')
ax.set_xlabel('Brand')
ax.set_ylabel('Selling Price (Lakhs)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

Now let us look at sample sizes for each categorical feature to understand class balance.

In [ ]:
# Value count bar charts for categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(['Fuel_Type', 'Transmission', 'Seller_Type', 'Brand']):
    ax = axes[i // 2][i % 2]
    if col == 'Brand':
        counts = df[col].value_counts().head(10)
        ax.set_title('Top 10 Brands by Count')
    else:
        counts = df[col].value_counts()
        ax.set_title(f'{col} -- Value Counts')
    counts.plot(kind='bar', color=sns.color_palette('Set2', len(counts)), ax=ax, edgecolor='black')
    ax.set_ylabel('Count')
    ax.set_xlabel(col)
    ax.tick_params(axis='x', rotation=45)
    # Add count labels on bars
    for j, v in enumerate(counts):
        ax.text(j, v + 0.5, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**Observations:**
- **Fuel Type:** Diesel cars tend to command higher resale prices than petrol. CNG cars (if present) are a small minority.
- **Transmission:** Automatic transmission cars have higher median prices, though manual dominates in volume.
- **Seller Type:** Dealer-listed cars show slightly higher and more consistent pricing than individual sellers.
- **Brand:** Maruti dominates the dataset in volume. Premium brands show higher median prices but with wider spread.

---
## 5. Key Insights

Based on this exploratory analysis, the following findings will guide feature engineering and model selection:

1. **Selling Price is right-skewed.** A log transformation significantly reduces skewness and should be considered as the target transformation during modelling (especially for linear models).

2. **Present Price is the strongest predictor.** It shows the highest positive correlation with Selling Price -- cars that cost more new also sell for more used. This single feature likely carries the most predictive power.

3. **Car Age drives depreciation.** Older cars sell for less. The negative relationship is clear, and Car Age will be a key feature in any model.

4. **Kilometres Driven has a weak negative signal.** While higher-mileage cars do tend to sell cheaper, the relationship is noisy. This feature may benefit from binning or interaction with Car Age.

5. **Fuel Type and Transmission matter.** Diesel and automatic cars command premium resale prices. These categorical features should be encoded and included in the model.

6. **Class imbalance in categories.** The dataset is dominated by Petrol + Manual + Dealer combinations, and by Maruti as a brand. Models should be evaluated carefully on minority categories to avoid bias toward the majority class.